In [ ]:
import pandas as pd
import sys
import re
!{sys.executable} -m pip install openpyxl

In [ ]:
archivo = "C:\\Users\\User\\Downloads\\1 AlbertoLineroGo Informe Ene 2024.xlsx"

df = pd.read_excel(
    archivo,
    sheet_name="BCE TERCERO"
)

In [ ]:
pd.read_excel(archivo,
    sheet_name="BCE TERCERO")

In [ ]:
# Mes y año
encabezado = pd.read_excel(archivo, nrows=7, header=None,
    sheet_name="BCE TERCERO")
texto = encabezado.iloc[4,0]

match = re.search(r"De\s+(\w+)\s+(\d{4})", texto)
if match:
    mes, año = match.groups()
    año = int(año)

df = pd.read_excel(archivo, skiprows=7,
    sheet_name="BCE TERCERO")

df["Año"] = año
df["Mes"] = mes

df.head()

In [ ]:
df = df.drop(columns=["Nivel", "Transaccional", "Identificación", "Sucursal"])

In [ ]:
df["Nombre tercero"] = df["Nombre tercero"].str.title()

In [ ]:
cols_monetarias = ["Saldo inicial", "Movimiento débito", "Movimiento crédito", "Saldo final"]
df[cols_monetarias] = df[cols_monetarias].apply(pd.to_numeric, errors="coerce")

In [ ]:
df["Largo"] = df["Código cuenta contable"].apply(lambda x: len(str(int(x))) if pd.notna(x) else 0)



In [ ]:
df = df[["Año", "Largo", "Mes", "Código cuenta contable", "Nombre cuenta contable",
         "Nombre tercero", "Saldo inicial", "Movimiento débito", "Movimiento crédito", "Saldo final"]]


In [ ]:
df["Cuenta"] = df["Código cuenta contable"].astype(str).str[0].map({
    "1": "Activo",
    "2": "Pasivo",
    "3": "Patrimonio"
}).fillna("")

In [ ]:
df = df[["Año", "Mes", "Largo", "Cuenta", "Código cuenta contable", "Nombre cuenta contable",
         "Nombre tercero", "Saldo inicial", "Movimiento débito", "Movimiento crédito", "Saldo final"]]

In [ ]:
df = df[df["Cuenta"] != ""]

In [ ]:
df["Dato Financiero"] = df.apply(
    lambda x: x["Saldo final"] if str(x["Código cuenta contable"]).startswith("1")
              else x["Saldo final"] * -1,
    axis=1
)

In [ ]:
df["Dato Financiero"] = pd.to_numeric(df["Dato Financiero"], errors="coerce")

In [ ]:
df["Puc & Descripción"] = df.apply(
    lambda row: (
        f"{int(row['Código cuenta contable'])} {row['Nombre cuenta contable']}"
        if pd.notna(row["Código cuenta contable"]) else f"{row['Nombre cuenta contable']}"
    ),
    axis=1
)



In [ ]:
df = df[["Año", "Mes", "Largo", "Cuenta", "Código cuenta contable", "Nombre cuenta contable",
         "Nombre tercero", "Puc & Descripción", "Saldo inicial", "Movimiento débito",
         "Movimiento crédito", "Saldo final", "Dato Financiero"]]

In [ ]:
print(df.head())

In [ ]:
# Exportar a excel
df.to_excel(
    r"C:\Users\User\Downloads\limpiado.xlsx",  
    index=False                                      
)
